# ATHLETIX AI: Empirical Dataset Verification & Exploratory Data Analysis 🏈⚡
> **Tagline**: *Beyond Speed. Understanding Athletic Potential.*  
> **Primary Research Question**: *Can advanced movement analytics identify NFL prospects whose athletic potential is underestimated by traditional Combine measurements?*

---

## Objectives of Phase 1 EDA
1. **Empirical Verification**: Audit actual row counts, schemas, and missing value representations across the 7 competition files.
2. **Memory Preservation (8 GB RAM)**: Leverage Polars lazy scanning (`pl.scan_csv`) to process millions of tracking observations without RAM exhaustion.
3. **Entity Resolution**: Verify universal cross-table linking via `nfl_id`.
4. **Kinematic Sanity Checks**: Differentiate between `PLAYER` vs `BALL` movement in combine tracking, and inspect game tracking coordinate bounds.
5. **Opt-Out Bias Analysis**: Quantify missingness in traditional Combine agility drills vs continuous sensor tracking.

In [ ]:
import sys
from pathlib import Path
import polars as pl
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Ensure project root is in path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
print(f"Polars Version: {pl.__version__}")
print(f"Data Directory: {DATA_DIR}")

## 1. Dataset Inventory & Null String ('NA') Discovery

In [ ]:
inventory_df = pd.read_csv(PROJECT_ROOT / "reports" / "actual_dataset_inventory.csv")
inventory_df

## 2. Combine Traditional Results & Severe Opt-Out Bias Analysis

Evaluating the high missingness rate across traditional Combine agility drills (`three_cone`, `short_shuttle`, `bench_reps`).

In [ ]:
# Load combine results with null_values=['NA', '']
combine_res = pl.read_csv(DATA_DIR / "combine_results.csv", null_values=["NA", ""])
print(f"Total Prospects in Combine Results: {len(combine_res)}")

missing_rates = combine_res.select([
    (pl.col(c).null_count() / len(combine_res) * 100).round(1).alias(c)
    for c in ["forty", "ten_yd_split", "vertical", "broad_jump", "three_cone", "short_shuttle", "bench_reps"]
]).to_pandas()

print("Opt-out / Missing percentage in traditional drills (%):")
display(missing_rates)

## 3. High-Frequency Combine Sensor Tracking (10 Hz)

Filtering between `entity_type == 'PLAYER'` vs `entity_type == 'BALL'` to inspect genuine human kinematics.

In [ ]:
combine_trk_lf = pl.scan_csv(DATA_DIR / "combine_tracking.csv")

# Compare PLAYER vs BALL velocities
entity_breakdown = combine_trk_lf.group_by("entity_type").agg([
    pl.len().alias("frame_count"),
    pl.col("s").max().alias("max_speed_yds"),
    (pl.col("s").max() * 2.04545).round(1).alias("max_speed_mph"),
    pl.col("s").mean().round(2).alias("mean_speed_yds"),
    pl.col("a").max().round(2).alias("max_accel_yds2"),
]).collect().to_pandas()

display(entity_breakdown)

## 4. Game Tracking Ingestion & Kinematic Distribution (2023 Season)

Lazy scan of 3,665,443 tracking frames from `game_tracking_2023.csv` for the 156 tracked rookie prospects.

In [ ]:
game_trk_lf = pl.scan_csv(DATA_DIR / "game_tracking_2023.csv")

g_stats = game_trk_lf.select([
    pl.len().alias("total_frames"),
    pl.col("nfl_id").n_unique().alias("unique_players"),
    pl.col("s").quantile(0.95).round(2).alias("speed_p95"),
    pl.col("s").quantile(0.99).round(2).alias("speed_p99"),
    pl.col("s").max().round(2).alias("max_game_speed"),
    (pl.col("s").max() * 2.04545).round(1).alias("max_game_speed_mph"),
    pl.col("a").max().round(2).alias("max_game_accel"),
]).collect().to_pandas()

display(g_stats)

## 5. Player Career Outcomes & Exposure Controls

Inspecting ground truth career indicators while controlling for draft cohort exposure (2023 vs 2024 vs 2025).

In [ ]:
players_df = pl.read_csv(DATA_DIR / "players.csv", null_values=["NA", ""])
career_df = pl.read_csv(DATA_DIR / "player_career_successes.csv", null_values=["NA", ""])

cohort_analysis = players_df.join(career_df, on="nfl_id").group_by("draft_year").agg([
    pl.len().alias("prospect_count"),
    pl.col("career_games_active").mean().round(1).alias("avg_active_games"),
    pl.col("career_games_started").mean().round(1).alias("avg_starts"),
    pl.col("career_offensive_snaps").mean().round(1).alias("avg_off_snaps"),
    pl.col("career_defensive_snaps").mean().round(1).alias("avg_def_snaps"),
    pl.col("pro_bowl_original_ballot").sum().alias("total_pro_bowls")
]).sort("draft_year").to_pandas()

display(cohort_analysis)